In [20]:
import pandas as pd

from legacy.old_model_generation.consts_dataframe import CANONICAL_GENE
from notebooks.consts import *
from notebooks.preprocessing import preprocess_aso_data

In [21]:
all_data = preprocess_aso_data(PROCESSED_CSV)

Adding cell line depmap name proxy
Adding a depmap column
Preprocessing complete. Final valid rows: 18153


In [22]:
from tauso.data.data import load_db

db = load_db()

In [23]:
from tauso.common.gtf import filter_gtf_genes
import os
from tauso.data.data import get_data_dir
from tauso.data.consts import CELL_LINE_DEPMAP
from tauso.features.codon_usage.find_cai_reference import load_cell_line_gene_expression
from tauso.data.consts import CANONICAL_GENE

data_dir = get_data_dir()
expression_dir = os.path.join(data_dir, "processed_expression")

depmap_ids = list(set(all_data[CELL_LINE_DEPMAP]))

target_genes = all_data[CANONICAL_GENE].unique().tolist()
target_genes.append('RNASEH1')

# 2. Run the Optimized Loader
#    This loads the DB/FASTA once, filters for valid genes, and enriches sequences.
transcriptomes = load_cell_line_gene_expression(
    depmap_ids,
    target_genes,
    expression_dir=expression_dir,  # Directory containing expression CSVs
)

In [25]:
import pandas as pd

# 1. Flatten the dictionary into one DataFrame
dfs_to_concat = []
for key, df in transcriptomes.items():
    # Take only what we need: 'Gene' and 'expression_norm'
    temp = df[['Gene', 'expression_norm']].copy()

    # Assign the dictionary key to a column using your variable CELL_LINE_DEPMAP
    temp[CELL_LINE_DEPMAP] = key
    dfs_to_concat.append(temp)

# Create one master reference table
expression_master = pd.concat(dfs_to_concat, ignore_index=True)

# ---------------------------------------------------------
# 2. Get 'target_expression'
# Match row's CELL_LINE_DEPMAP -> CELL_LINE_DEPMAP
# Match row's CANONICAL_GENE -> 'Gene'
# ---------------------------------------------------------
all_data = all_data.merge(
    expression_master,
    left_on=[CELL_LINE_DEPMAP, CANONICAL_GENE], # Variable column names from all_data
    right_on=[CELL_LINE_DEPMAP, 'Gene'],         # Hardcoded column names from transcriptomes
    how='left'
)

# Rename the value column to target_expression
all_data = all_data.rename(columns={'expression_norm': 'target_expression'})

# Drop the 'Gene' column that came from expression_master (since we have CANONICAL_GENE)
all_data = all_data.drop(columns=['Gene'])

# ---------------------------------------------------------
# 3. Get 'rnase_expression' (Hardcoded Gene 'RNASEH1')
# ---------------------------------------------------------
# Filter the master list for RNASEH1 only
rnase_vals = expression_master[expression_master['Gene'] == 'RNASEH1']

# Merge only on CELL_LINE_DEPMAP
all_data = all_data.merge(
    rnase_vals[[CELL_LINE_DEPMAP, 'expression_norm']],
    on=CELL_LINE_DEPMAP,
    how='left',
    suffixes=('', '_rnase') # Protect against name collisions
)

# Rename the value column to rnase_expression
all_data = all_data.rename(columns={'expression_norm': 'rnase_expression'})

In [27]:
from notebooks.features.feature_extraction import save_feature

for feature in ['rnase_expression', 'target_expression']:
    save_feature(all_data, feature, version='v2')

Saved feature: rnase_expression
Saved feature: target_expression
